# Fusion Architecture Design Ablation Study

This analytical review compares multi-modal signal fusion operators designed to map continuous scalar break probabilities into dense textual query features. The evaluation operates at a fixed hidden dimension ($m=64$) using the definitive **Gold M signal** under the **Trainable Sequential Linear Projection** head.

### Fusion Operator Configurations
- **Concat**: Direct low-level vector concatenation mapping continuous probability features into shared sequence contexts.
- **Gated**: Element-wise multiplicative gating mechanisms learning continuous mask maps over semantic vectors.
- **FiLM (Scale)**: Feature-wise Linear Modulation applying affine input scaling ($\gamma$) without positional shifting.
- **FiLM (Full)**: Comprehensive affine modulation scaling ($\gamma$) and shifting ($\beta$) matching the underlying query feature manifold.

### Verification Parity Protocols
- **Multi-Seed Averaging**: Every architecture reports metrics averaged across an identical 5-seed cohort (`7, 13, 21, 42, 100`) to guarantee robust comparisons.
- **Zero Data-Leakage**: Stratification splits prevent literal token expression sharing across training and evaluation boundaries.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Premium tailored visual aesthetics: minimal grids, high resolution, soft tones
plt.style.use('default')
sns.set_theme(style="white", palette="muted")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

summary_path = Path("output/summary.json")

if summary_path.exists():
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary = json.load(f)
    print("=== FUSION ARCHITECTURE PAYLOAD INGESTED ===")
    print(f"Evaluated Topologies Captured: {len(summary.get('aggregate', []))}")
else:
    summary = None
    print("[Status] Target fusion summary payload currently unavailable.")
    print(f"Expected Path: {summary_path}")
    print("Execute 'python run.py' to populate modular validation iterations.")

In [ ]:
if summary:
    rows = []
    for agg in summary.get("aggregate", []):
        rows.append({
            "Fusion Architecture": agg["group_label"],
            "Dev nDCG (%)": agg["best_dev"]["ndcg"] * 100.0,
            "Test nDCG (%)": agg["test"]["ndcg"] * 100.0,
            "Cross-Lingual nDCG (%)": agg["xeval"]["ndcg"] * 100.0,
            "Test Top-1 Exact (%)": agg["test"]["top1"] * 100.0,
            "Avg Optimal Epoch": agg["best_epoch"]
        })
        
    df = pd.DataFrame(rows)
    
    print("\n=== FUSION TOPOLOGY PERFORMANCE MATRIX ===")
    display(df.round(2))

### Downstream Listwise Alignment (Dev vs. Test nDCG)
Compares multi-split ranking parity across primary validation sets, excluding auxiliary cross-lingual runs to ensure focused comparisons.

In [ ]:
if summary:
    # Focus strictly on primary validation splits for high-impact visual clarity
    melted_df = df.melt(id_vars=["Fusion Architecture"], 
                        value_vars=["Dev nDCG (%)", "Test nDCG (%)"],
                        var_name="Evaluation Set", value_name="nDCG Score (%)")
                        
    fig, ax = plt.subplots(figsize=(10, 5.5))
    
    # Implements ultra-clean soft background grid layout
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='#cccccc')
    ax.set_axisbelow(True)
    
    sns.barplot(data=melted_df, x="Fusion Architecture", y="nDCG Score (%)", 
                hue="Evaluation Set", palette=["#2b8cbe", "#31a354"], ax=ax)
                
    sns.despine(left=True, bottom=False)
    ax.set_title("Listwise Ranking Parity across Primary Evaluation Splits", fontweight="bold", fontsize=13, pad=15)
    
    # Generates custom upper axis bounds to grant visual headroom for annotations
    y_max = melted_df["nDCG Score (%)"].max()
    ax.set_ylim(melted_df["nDCG Score (%)"].min() * 0.92, y_max * 1.08)
    ax.set_xlabel("")
    ax.set_ylabel("Accuracy (nDCG %)", fontweight="bold", fontsize=11)
    ax.tick_params(axis='x', labelsize=11)
    ax.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right", fontsize=10)
    
    # Premium label spacing and elevated positioning
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.2f}%", (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', xytext=(0, 6), textcoords='offset points', 
                         fontweight='bold', fontsize=10, color='#333333')
                         
    plt.tight_layout()
    plt.show()

### Exact Top-1 Retrieval Precision (Test Set)
Evaluates the absolute target precision capacity of each operator to place ground-truth target configurations exactly at rank 1.

In [ ]:
if summary:
    fig, ax = plt.subplots(figsize=(9, 5))
    
    ax.grid(axis='y', linestyle='--', alpha=0.4, color='#cccccc')
    ax.set_axisbelow(True)
    
    # Premium isolated categorical feature visual layout
    sns.barplot(data=df, x="Fusion Architecture", y="Test Top-1 Exact (%)", 
                palette="rocket", hue="Fusion Architecture", legend=False, ax=ax)
                
    sns.despine(left=True, bottom=False)
    ax.set_title("Test Set Exact Top-1 Alignment Precision", fontweight="bold", fontsize=13, pad=15)
    
    y_max = df["Test Top-1 Exact (%)"].max()
    ax.set_ylim(df["Test Top-1 Exact (%)"].min() * 0.88, y_max * 1.10)
    ax.set_xlabel("")
    ax.set_ylabel("Top-1 Match (%)", fontweight="bold", fontsize=11)
    ax.tick_params(axis='x', labelsize=11)
    
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f"{height:.2f}%", (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom', xytext=(0, 6), textcoords='offset points', 
                         fontweight='bold', fontsize=10.5, color='#333333')
                         
    plt.tight_layout()
    plt.show()